# Assignment 1: Weather Agent Debugging

This notebook documents the bugs found in the LangGraph weather agent, the fixes applied, and test scenarios used to verify the final behavior.

## Bugs found and fixed

1. `main.py` used `_main_` instead of `__main__`, so the program entrypoint did not run. This was already fixed before this notebook was created.
2. `fetch_location_data` validated `country`, but the formatter expects `country_name`, which is the field returned by `ipapi.co` for the full country name.
3. `fetch_location_data` replaced the real API response with an empty dictionary, which caused the weather node to fail because latitude and longitude were missing.
4. `fetch_weather_data` contained a misleading comment saying Open-Meteo requires an API key. The public forecast endpoint used here does not require one.
5. `generate_weather_info` commented out `wind_unit` and printed the literal text `wind_unit` instead of the actual unit.
6. `generate_weather_info` contained text encoding artifacts for bullets and the fallback Celsius unit. The output now uses ASCII bullets and a safe fallback unit.
7. `requirements.txt` did not include `requests`, even though the nodes import and use it directly.
8. `requirements.txt` was stored as UTF-16. It was normalized to UTF-8 so normal Python and diff tooling can read it cleanly.

## Import and compile the graph

If this cell imports successfully, LangGraph can build the weather agent.

In [ ]:
from graph import weather_agent

type(weather_agent).__name__

## Mocked API helpers

The assignment requires testing different scenarios. These tests mock the external APIs so the notebook is repeatable and does not depend on the current network, IP address, or weather.

In [ ]:
from unittest.mock import patch


class MockResponse:
    def __init__(self, payload):
        self.payload = payload

    def raise_for_status(self):
        return None

    def json(self):
        return self.payload


def location_payload(city="Kuala Lumpur", region="Kuala Lumpur", country="Malaysia", offset="+08:00"):
    return {
        "city": city,
        "region": region,
        "country_name": country,
        "latitude": 3.139,
        "longitude": 101.6869,
        "utc_offset": offset,
        "timezone": "Asia/Kuala_Lumpur",
    }


def weather_payload(temperature, weathercode, is_day, windspeed=8.5):
    return {
        "current_weather_units": {
            "temperature": "C",
            "windspeed": "km/h",
        },
        "current_weather": {
            "time": "2026-05-02T03:30",
            "temperature": temperature,
            "windspeed": windspeed,
            "winddirection": 120,
            "is_day": is_day,
            "weathercode": weathercode,
        },
    }


def run_agent_with_payloads(location, weather, name="Aina"):
    responses = [MockResponse(location), MockResponse(weather)]
    initial_state = {
        "name": name,
        "location_data": None,
        "weather_data": None,
        "weather_info": None,
    }

    with patch("components.nodes.requests.get", side_effect=responses):
        return weather_agent.invoke(initial_state)

## Scenario 1: clear daytime weather

In [ ]:
final_state = run_agent_with_payloads(
    location_payload(),
    weather_payload(temperature=29.0, weathercode=0, is_day=1),
)

print(final_state["weather_info"])

Expected result: the report greets the user, includes the location, describes clear sky, classifies `29.0C` as warm, and shows wind speed with `km/h`.

## Scenario 2: rainy nighttime weather

In [ ]:
final_state = run_agent_with_payloads(
    location_payload(city="London", region="England", country="United Kingdom", offset="+00:00"),
    weather_payload(temperature=7.0, weathercode=63, is_day=0, windspeed=14.0),
)

print(final_state["weather_info"])

Expected result: the report uses an evening greeting, describes moderate rain, classifies `7.0C` as cold, and displays the wind unit correctly.

## Scenario 3: malformed location API response

In [ ]:
bad_location = location_payload()
bad_location.pop("country_name")

try:
    run_agent_with_payloads(
        bad_location,
        weather_payload(temperature=20.0, weathercode=2, is_day=1),
    )
except Exception as exc:
    print(exc)

Expected result: the agent raises a clear validation error explaining that `country_name` is missing.

## Optional live run

The command below calls the real location and weather APIs. It may fail in restricted environments, but it should work when outbound HTTPS requests are allowed.

```bash
python main.py
```